In [1]:
import json
import pathlib

import pandas as pd

In [2]:
from kebab.utils.dataset.wikidata.wikidata_type_hierarchy_extractor import WikidataTypeHierarchyExtractor

Sample element from the dataset
---

In [3]:
# load the Wikidata hierarchy
hierarchy_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "Wikidata"
    / "Type Hierarchy"
    / "2025-01-30"
    / "wikidata_type_hierarchy.jsonl"
)

graph = {}
with open(hierarchy_path, encoding="utf-8") as f:
    for line in f:
        node = json.loads(line.strip())
        graph[node["id"]] = node

print(f"Type hierarchy contains {len(graph):,d} nodes")
print("Sample node:")
graph["Q5"]

Type hierarchy contains 121,997 nodes
Sample node:


{'id': 'Q5',
 'merged_ids': ['Q5'],
 'redirect_from_ids': [],
 'name': 'human',
 'descriptions': ['any member of Homo sapiens, unique extant species of the genus Homo'],
 'aliases': ['individual Homo sapiens',
  'modern humans',
  'people',
  'human',
  'person',
  'humans',
  'individual human',
  'nonfictional human',
  'non-fictional human',
  'human being',
  'man'],
 'parents': ['Q164509', 'Q110551885', 'Q26401003', 'Q154954'],
 'children': ['Q254917',
  'Q24760463',
  'Q224952',
  'Q1020994',
  'Q97322761',
  'Q7569',
  'Q26513',
  'Q18093576',
  'Q2907480',
  'Q16172115',
  'Q1378555',
  'Q106155',
  'Q29863201',
  'Q40889758',
  'Q5054938',
  'Q107662846',
  'Q84048850',
  'Q9584157',
  'Q844586',
  'Q104716993',
  'Q20643955',
  'Q18093573',
  'Q84048852',
  'Q18121791',
  'Q319604',
  'Q2072081'],
 'ref_count': 11112520,
 'subgraph_size': 116,
 'subgraph_height': 8,
 'subgraph_ref_count': 11114138}

Main root node (Entity)
---

In [4]:
graph["Q35120"]

{'id': 'Q35120',
 'merged_ids': ['Q35120'],
 'redirect_from_ids': [],
 'name': 'entity',
 'descriptions': ['anything that can be considered, discussed, or observed'],
 'aliases': ['object', 'entity', 'thing'],
 'parents': [],
 'children': ['Q7048977',
  'Q488383',
  'Q3826351',
  'Q25047676',
  'Q30241068',
  'Q18706315',
  'Q121770302',
  'Q115095765',
  'Q2995644',
  'Q3885844',
  'Q99527517',
  'Q103940464',
  'Q31464082',
  'Q35825432',
  'Q64728693',
  'Q15989253',
  'Q42503289',
  'Q378078',
  'Q120725535',
  'Q15893266',
  'Q24238356',
  'Q23958946'],
 'ref_count': 3,
 'subgraph_size': 99872,
 'subgraph_height': 50,
 'subgraph_ref_count': 94456721}

Nodes with the most references
---


In [5]:
# nodes with the most references
df = pd.DataFrame(
    [(node["id"], node["name"], node["ref_count"], node["descriptions"]) for node in graph.values()],
    columns=["id", "name", "ref_count", "descriptions"],
)

df = df.sort_values("ref_count", ascending=False).reset_index(drop=True)
df.head(10)

,id,name,ref_count,descriptions
0,Q13442814,scholarly article,42725566,"[article in an academic publication, usually p..."
1,Q5,human,11112520,"[any member of Homo sapiens, unique extant spe..."
2,Q16521,taxon,3749506,"[group of one or more organism(s), which a tax..."
3,Q4167836,Wikimedia category,3464955,[use with 'instance of' (P31) for Wikimedia ca...
4,Q60535861,survey article,2096917,[document containing a comprehensive overview ...
5,Q113145171,type of chemical entity,1274348,[Wikidata metaclass that covers physical entit...
6,Q7187,gene,1224818,[basic physical and functional unit of heredity]
7,Q8054,protein,1006995,[biomolecule or biomolecule complex largely co...
8,Q4167410,Wikimedia disambiguation page,806428,[type of wiki page usually in main namespace (...
9,Q1792880,Kuppe,729230,"[elevated place with circular crosssection, wi..."


Nodes with the highest merge number
---

These are nodes that have been merged with other nodes due to same-to-be-the-same-as links or parent relationship cycles    

In [6]:
# nodes with largest merged node count
df = pd.DataFrame(
    [
        (node["id"], node["name"], node["aliases"], len(node["merged_ids"]), node["ref_count"])
        for node in graph.values()
    ],
    columns=["id", "name", "aliases", "merged_count", "ref_count"],
)

df = df.sort_values("merged_count", ascending=False).reset_index(drop=True)
df.head(10)

,id,name,aliases,merged_count,ref_count
0,Q1157107,div (mythology),"[genies (plural), S̲h̲aiṭān, chort, Div, div (...",12,218
1,Q111165099,nagori,"[gampong, kampung, pekon, village (Indonesia),...",11,85614
2,Q106458883,state,"[state, province, district, territory, parish,...",6,0
3,Q112635088,třída,"[Ave., alameda, tree-lined street, třída, Stre...",6,14112
4,Q11348,function,"[right unique relation, partial function, righ...",6,197
5,Q12341378,kneipe,"[kabacke, tavern, korchma, kneipe, kabak, kaba...",6,227
6,Q11971378,associate professor,"[assistant professor, Associate Prof., associa...",5,0
7,Q126838043,,"[, land worker, peasant, farm hand, field hand...",5,6
8,Q2022868,environmental effects,"[environmental effects, human-induced hazard, ...",5,95
9,Q1589434,magister degree,"[laurea magistrale, M.A., Magister, master's d...",5,174


Root nodes with the highest combined reference count
---

These are the top-level nodes in the hierarchy that have the most references when considering all their children nodes.

In [7]:
# largest root nodes
root_node_ids = WikidataTypeHierarchyExtractor.get_root_entities(graph)
print(f"Root nodes: {len(root_node_ids):,d}")

root_nodes = [graph[node_id] for node_id in root_node_ids]

df = pd.DataFrame(
    [(node["id"], node["name"], node["subgraph_ref_count"], node["descriptions"]) for node in root_nodes],
    columns=["id", "name", "subgraph_ref_count", "descriptions"],
)

df = df.sort_values("subgraph_ref_count", ascending=False).reset_index(drop=True)
df.head(20)

Root nodes: 21,650


,id,name,subgraph_ref_count,descriptions
0,Q35120,entity,94456721,"[anything that can be considered, discussed, o..."
1,Q130370861,"law, advocacy and politics",1917318,[activity of nonprofit-organizations]
2,Q175661,action theory,91377,[area in philosophy concerned with theories ab...
3,Q25689702,,41572,[Wikimedia permanent duplicate item]
4,Q125598647,continuant type,18420,[]
5,Q96622155,reference,14392,[reference to another work]
6,Q117769285,resources industry,14271,[subconcept of industries]
7,Q151411,designation,3753,[representation of a term by linguistic or oth...
8,Q131164,Web 2.0,1553,[World Wide Web sites that use technology beyo...
9,Q26220,Vitaceae,1528,[family of plants]


Leaf nodes with the highest reference count
---

In [8]:
# largest leaf nodes
leaf_node_ids = WikidataTypeHierarchyExtractor.get_leaf_entities(graph)
print(f"Leaf nodes: {len(leaf_node_ids):,d}")

leaf_nodes = [graph[node_id] for node_id in leaf_node_ids]
df = pd.DataFrame(
    [(node["id"], node["name"], node["subgraph_ref_count"], node["descriptions"]) for node in leaf_nodes],
    columns=["id", "name", "subgraph_ref_count", "descriptions"],
)

df = df.sort_values("subgraph_ref_count", ascending=False).reset_index(drop=True)
df.head(20)

Leaf nodes: 85,635


,id,name,subgraph_ref_count,descriptions
0,Q113145171,type of chemical entity,1274348,[Wikidata metaclass that covers physical entit...
1,Q871232,editorial,511820,[journalism genre]
2,Q59199015,group of stereoisomers,148039,[set of several stereoisomers]
3,Q115595777,taxonomy template,140371,[use with P31 for templates that are a subpage...
4,Q61443690,branch post office,129183,[type of post office of India]
5,Q815382,meta-analysis,109642,[statistical method that summarizes data from ...
6,Q7604686,UK Statutory Instrument,99930,[type of secondary legislation in the United K...
7,Q111165099,nagori,85614,"[, fourth-level administrative area under the ..."
8,Q97042318,sekolah dasar,79483,[Indonesian elementary school]
9,Q22808320,Wikimedia human name disambiguation page,76595,[Wikimedia disambiguation page for humans with...
